# Annual change in Thailand's international tourism receipts from a theme park

**Purpose.** This notebook applies the route rule that was fixed in advance to choose between the importance-weighted optimal-transport route and the ambient fallback route, and then reports the annual change in international tourism receipts that the opening of a park of the proposed cost would bring to the target economy. The rule compares leave-one-economy-out prediction errors in which the importance weights are relearned inside every fold. The change is expressed in percentage points of GDP and in percent of the counterfactual receipts, scaled to US dollars and baht with the baseline of 2019 and of 2024, and accompanied by the minimum detectable effect, the probability of detection and a reference comparison with the claims that officials have made.

**Input files.** In the results directory: case_effects.csv, case_features.csv, case_features_imputed.csv, importance_diagnostics.json, loco_summary.csv, loco_table.csv, loco_fold_weights.csv, thailand_transport.json, thailand_transport_draws.csv and thailand_null_draws.csv. The run context also supplies the Thailand baseline and the proposal file.

**Output files.** In the results directory: route_decision.json, ambient_loeo.csv, thailand_impact.csv and thailand_summary.json. In the figures directory: impact_routes.png and impact_scaling_law.png.

**How to run.** On the desktop, set the environment variable DTT_RUN_REAL=1 and execute the notebook with `jupyter nbconvert --to notebook --execute notebooks/05_thailand_impact.ipynb`. The simulated check sets DTT_WORLD=sim together with DTT_SIM_DIR, DTT_RESULTS_DIR and DTT_FIGURE_DIR, which name empty temporary directories, and needs the earlier notebooks to have run in the same mode.

**Run time.** About half a minute on the real data and about 5 seconds in the simulated mode.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
_root = next(p for p in (_here, *_here.parents) if (p / "src" / "dtt" / "__init__.py").is_file())
sys.path.insert(0, str(_root / "src"))

from dtt import workflow

ctx = workflow.setup_run("05_thailand_impact")
ctx.describe()

## Route decision

The primary route is chosen before any estimate for the target economy is shown. The rule was fixed in advance and is applied without change: the primary route is the importance-weighted optimal-transport route if and only if the importance diagnostics report a usable model, the leave-one-economy-out RMSE of ot_weighted is at most the stated multiple of that of equal and of ot_uniform, and the validated sample has at least the stated number of episodes. In the validation the importance weights are relearned inside every fold from the training economies only, so the RMSE of ot_weighted does not use the left-out economy to choose its weights. The limits are printed below. In every other case the primary route is the ambient route. The importance route and the ambient routes are reported side by side whichever route is primary.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from scipy import stats

from dtt import impact, meta

NOTEBOOK_START = time.perf_counter()
ctx.require(
    "case_effects",
    "case_features",
    "case_features_imputed",
    "importance_diagnostics.json",
    "loco_summary",
    "loco_table",
    "loco_fold_weights",
    "thailand_transport.json",
    "thailand_transport_draws",
    "thailand_null_draws",
)
IMPORTANCE_KEY = "att_pp|primary"
importance_all = ctx.read_json("importance_diagnostics")
if IMPORTANCE_KEY not in importance_all:
    raise KeyError(f"importance_diagnostics.json has no entry {IMPORTANCE_KEY!r}; its entries are {sorted(importance_all)}")
importance_diagnostics = importance_all[IMPORTANCE_KEY]
IMPORTANCE_USABLE = bool(importance_diagnostics["usable"])
loco_summary = ctx.read_table("loco_summary", index_col="method")
loco_table = ctx.read_table("loco_table")
fold_weights = ctx.read_table("loco_fold_weights")
effects = ctx.read_table("case_effects")
n_primary = int(effects["sample_primary"].astype(bool).sum())
validated = loco_table[loco_table["method"] == "ot_weighted"]
n_validated = int(validated["case"].nunique())
n_validated_economies = int(validated["group"].nunique())
fold_fitted = fold_weights.groupby("held_out_group")["fitted"].all()
n_folds = int(fold_fitted.size)
n_uniform_folds = int((~fold_fitted).sum())
rules = dict(impact.DEFAULT_ROUTE_RULES)

decision = impact.decide_route({"usable": IMPORTANCE_USABLE}, loco_summary, n_validated)

print("Route rule, fixed in advance and applied without change")
print("  The primary route is ot_importance if and only if all three criteria hold, and ambient otherwise.")
print("  1. The importance diagnostics report a usable model.")
print(
    f"  2. The leave-one-economy-out RMSE of ot_weighted, with the importance weights relearned inside every fold, is at most {rules['rmse_ratio_equal']:g} times that of equal "
    f"and at most {rules['rmse_ratio_uniform']:g} times that of ot_uniform."
)
print(f"  3. The validated sample has at least {rules['min_cases']:g} episodes.")
print(f"\nInputs: importance entry {IMPORTANCE_KEY!r} with usable = {IMPORTANCE_USABLE}")
for reason in importance_diagnostics.get("reasons", []):
    print(f"  importance diagnostics: {reason}")
print(
    f"Primary episodes flagged: {n_primary}; validated: {n_validated} episodes in {n_validated_economies} economies; "
    f"importance weights relearned in {n_folds - n_uniform_folds} of {n_folds} folds and uniform in {n_uniform_folds}"
)
print("Leave-one-economy-out RMSE (percentage points of GDP):")
print(loco_summary.loc[["ot_weighted", "ot_uniform", "equal"], ["rmse"]].round(4).to_string())
print("\nCriteria met:", {name: bool(flag) for name, flag in decision.passed.items()})
print(decision.details)
print(f"\nPrimary route: {decision.primary}")

route_record = {
    "primary": decision.primary,
    "passed": {name: bool(flag) for name, flag in decision.passed.items()},
    "details": decision.details,
    "rule": {
        "importance_usable_required": True,
        "rmse_ratio_equal": rules["rmse_ratio_equal"],
        "rmse_ratio_uniform": rules["rmse_ratio_uniform"],
        "min_cases": rules["min_cases"],
        "fixed_in_advance": True,
    },
    "inputs": {
        "importance_key": IMPORTANCE_KEY,
        "importance_usable": IMPORTANCE_USABLE,
        "importance_reasons": list(importance_diagnostics.get("reasons", [])),
        "loco_rmse": {str(method): float(value) for method, value in loco_summary["rmse"].items()},
        "n_primary_episodes": n_primary,
        "n_validated_episodes": n_validated,
        "n_validated_economies": n_validated_economies,
        "validation_weights": "importance relearned in every fold from the training economies",
        "n_folds": n_folds,
        "n_folds_uniform_weights": n_uniform_folds,
    },
}
saved_route = ctx.save_json(route_record, "route_decision")

## Settings

The settings cell fixes every setting once. The flag FAST reduces only the number of random draws; the method is the same in every mode.

In [ ]:
# Settings (FAST changes the number of draws only)
FAST = ctx.mode == "sim"
N_DRAWS = 20000 if FAST else 100000  # predictive draws of each ambient estimator
N_DRAWS_FOLD = 4000 if FAST else 20000  # posterior draws in each leave-one-economy-out refit of A1
BASELINE_YEARS = (2019, 2024)  # baseline years that scale the effect to US dollars
REFERENCE_YEAR = 2019  # year of the exchange rate, of the GDP in the cost share and of the receipts window
WINDOW_YEARS = tuple(range(REFERENCE_YEAR - 2, REFERENCE_YEAR + 1))  # years of the receipts share, the window of the case features
PROBS = (0.05, 0.25, 0.5, 0.75, 0.95)  # quantile levels of the reported intervals
LEVEL = 0.9  # coverage of the Student prediction intervals of A2 and A3
RIDGE = 1.0  # ridge weight of A3, in units of the mean precision of one case on the standardised slopes
SELECTION_RATIO = 0.95  # A2 or A3 replaces A1 only if its RMSE is at most this multiple of the RMSE of A1
ALPHA = 0.05  # size of the detection threshold
POWER = 0.8  # target power of the minimum detectable effect
A3_FEATURES = ["capex_pct_gdp", "receipts_pct_gdp"]  # covariates of the meta-regression
SEED_META = ctx.seeds["meta"]

print(f"FAST = {FAST}")
print(f"Draws: {N_DRAWS} per ambient estimator, {N_DRAWS_FOLD} per refit of A1 in the validation; seed {SEED_META}")
print(f"Baseline years {BASELINE_YEARS}; reference year {REFERENCE_YEAR}; quantile levels {PROBS}")
print(f"A3 covariates {A3_FEATURES} with ridge weight {RIDGE:g}; A1 with the default priors of the module; prediction level {LEVEL:g}")
print(f"Detection: size {ALPHA}, power {POWER}")

## Helper functions

In [ ]:
def show(table: pd.DataFrame, digits: int = 3) -> None:
    """Print a table with a fixed number of decimals."""
    print(table.round(digits).to_string())


def quantile_dict(values: np.ndarray, probs: tuple[float, ...] = PROBS) -> dict[str, float]:
    """Quantiles of a sample as a dictionary with the keys ``q5``, ``q25`` and so on."""
    return {f"q{round(100 * p):g}": float(q) for p, q in zip(probs, np.quantile(values, probs))}


def student_draws(prediction: dict[str, float], df: int, n: int, rng: np.random.Generator, label: str = "") -> np.ndarray | None:
    """Draws of the predictive distribution of a regression estimator.

    Parameters
    ----------
    prediction : dict
        Entries ``mean``, ``lo``, ``hi`` and ``se_pred`` of the prediction for the target.
    df : int
        Degrees of freedom of the Student distribution.
    n : int
        Number of draws.
    rng : numpy.random.Generator
        Random generator.
    label : str
        Name of the estimator, used in the message when its interval is unavailable.

    Returns
    -------
    numpy.ndarray or None
        ``n`` draws with location ``mean``, scale ``se_pred`` and ``df`` degrees of freedom. ``None``, after a printed
        message, when ``df`` is below one or a limit of the prediction interval is not finite, because the interval of
        the estimator is then unavailable and the estimator is left out of the draws.
    """
    limits = np.array([prediction["lo"], prediction["hi"]], dtype=float)
    if df < 1 or not np.isfinite(limits).all():
        print(f"{label}: the prediction interval has {df} degrees of freedom and no finite limits, so it is unavailable and the estimator is left out of the draws")
        return None
    return float(prediction["mean"]) + float(prediction["se_pred"]) * rng.standard_t(df, size=n)


def predict_held_out(estimator: str, train: pd.DataFrame, test: pd.DataFrame, outcome: str, se_column: str) -> np.ndarray:
    """Point predictions of held-out episodes by one ambient estimator.

    Parameters
    ----------
    estimator : str
        ``A1`` (pooled predictive mean), ``A2`` (scaling law in the cost share of GDP) or ``A3`` (meta-regression).
    train : pandas.DataFrame
        Episodes that the estimator is fitted to.
    test : pandas.DataFrame
        Episodes to predict.
    outcome : str
        Column of the effect.
    se_column : str
        Column of the standard error of the effect.

    Returns
    -------
    numpy.ndarray
        One prediction per row of ``test``.
    """
    y = train[outcome].to_numpy(dtype=float)
    s = train[se_column].to_numpy(dtype=float)
    with warnings.catch_warnings():
        # The refits are used for their point predictions; the interval warnings of small training samples are not shown
        warnings.filterwarnings("ignore", message="The prediction interval has")
        if estimator == "A1":
            fit = meta.bayes_normal_hierarchical(y, s, n_draws=N_DRAWS_FOLD, seed=SEED_META)
            return np.full(len(test), float(fit.summary.loc["predictive", "mean"]))
        if estimator == "A2":
            fit = meta.scaling_law(y, s, train["capex_pct_gdp"], test[["capex_pct_gdp"]], level=LEVEL)
            return np.asarray(fit["prediction"]["mean"], dtype=float)
        if estimator == "A3":
            fit = meta.meta_regression(y, s, train[A3_FEATURES], x_new=test[A3_FEATURES], ridge=RIDGE, level=LEVEL)
            return np.asarray(fit["prediction"]["mean"], dtype=float)
    raise ValueError(f"unknown estimator {estimator!r}")


def fit_ambient(estimator: str, frame: pd.DataFrame, outcome: str, se_column: str, target: dict[str, float], seed: int, label: str = "") -> dict[str, object]:
    """Fit one ambient estimator to the primary sample and draw the predictive distribution for the target.

    Parameters
    ----------
    estimator : str
        ``A1``, ``A2`` or ``A3``.
    frame : pandas.DataFrame
        Primary episodes with the effect, its standard error and the covariates.
    outcome : str
        Column of the effect.
    se_column : str
        Column of the standard error of the effect.
    target : dict
        Covariates of the target economy, with the names of ``A3_FEATURES``; the module matches them to the covariates
        of the episodes by name.
    seed : int
        Seed of the random draws.
    label : str
        Name of the fit, used in the message when the interval is unavailable.

    Returns
    -------
    dict
        ``draws`` (predictive draws, ``None`` when the interval of the estimator is unavailable), ``mean``, ``lo`` and
        ``hi`` (point prediction and interval at level ``LEVEL`` for A2 and A3, quantiles of the draws for A1) and
        ``fit`` (the output of the module function).
    """
    y = frame[outcome].to_numpy(dtype=float)
    s = frame[se_column].to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    if estimator == "A1":
        fit = meta.bayes_normal_hierarchical(y, s, n_draws=N_DRAWS, seed=seed)
        draws = fit.draws_pred
        lo, hi = np.quantile(draws, [0.5 - LEVEL / 2.0, 0.5 + LEVEL / 2.0])
        return {"draws": draws, "mean": float(draws.mean()), "lo": float(lo), "hi": float(hi), "fit": fit}
    if estimator == "A2":
        fit = meta.scaling_law(y, s, frame["capex_pct_gdp"], target, level=LEVEL)
    elif estimator == "A3":
        fit = meta.meta_regression(y, s, frame[A3_FEATURES], x_new=target, ridge=RIDGE, level=LEVEL)
    else:
        raise ValueError(f"unknown estimator {estimator!r}")
    prediction = fit["prediction"]
    draws = student_draws(prediction, int(fit["df"]), N_DRAWS, rng, label)
    return {"draws": draws, "mean": float(prediction["mean"]), "lo": float(prediction["lo"]), "hi": float(prediction["hi"]), "fit": fit}


def scaled_summary(draws_pp: np.ndarray, draws_rel: np.ndarray, row: pd.Series) -> dict[str, dict[str, dict[str, float]]]:
    """Quantiles of the scaled effect of one baseline year, for both scalings.

    Parameters
    ----------
    draws_pp : numpy.ndarray
        Draws of the effect in percentage points of GDP.
    draws_rel : numpy.ndarray
        Draws of the effect in percent of the counterfactual receipts.
    row : pandas.Series
        Baseline year with ``gdp_usd_bn``, ``receipts_usd_bn`` and ``fx_thb_per_usd``.

    Returns
    -------
    dict
        For each scaling (``absolute``, ``relative``) the quantiles ``q5``, ``q50`` and ``q95`` of the annual change
        in US$ billion (``usd_bn``), in THB billion (``thb_bn``), as a share of the baseline receipts
        (``share_of_receipts_pct``) and as a share of the baseline GDP (``share_of_gdp_pct``).
    """
    scaled = impact.scale_effect(draws_pp, draws_rel, row)
    out: dict[str, dict[str, dict[str, float]]] = {}
    for scaling, usd, thb in (
        ("absolute", scaled["absolute_usd_bn"], scaled["absolute_thb_bn"]),
        ("relative", scaled["relative_usd_bn"], scaled["relative_thb_bn"]),
    ):
        levels = (0.05, 0.5, 0.95)
        q_usd = np.quantile(usd, levels)
        q_thb = np.quantile(thb, levels)
        keys = ("q5", "q50", "q95")
        out[scaling] = {
            "usd_bn": dict(zip(keys, map(float, q_usd))),
            "thb_bn": dict(zip(keys, map(float, q_thb))),
            "share_of_receipts_pct": dict(zip(keys, (float(100.0 * q / row["receipts_usd_bn"]) for q in q_usd))),
            "share_of_gdp_pct": dict(zip(keys, (float(100.0 * q / row["gdp_usd_bn"]) for q in q_usd))),
        }
    return out

## Target economy and baseline

The target economy is described by the Thailand baseline and the primary scenario of the proposal. The baseline loader checks that each baseline year used below has all five series and that every value is finite and positive; the receipts share must be present in every year of its window. The cost in baht is converted to US dollars with the exchange rate of the reference year, and the cost share of GDP uses the GDP of that year, which is the definition of the cost feature of the cases. The receipts share is the mean of the three years that end in the reference year, which is the window of the case features.

In [ ]:
baseline = impact.load_thailand_baseline(ctx.baseline_path, years=list(BASELINE_YEARS))
window_share = baseline.reindex(list(WINDOW_YEARS))["receipts_pct_gdp"]
assert window_share.notna().all(), f"the receipts share of GDP is missing in {[int(y) for y in window_share.index[window_share.isna()]]}"
scenario = ctx.primary_scenario()
fx_reference = float(baseline.loc[REFERENCE_YEAR, "fx_thb_per_usd"])
capex_usd_bn = float(scenario["capex_thb_bn"]) / fx_reference
gdp_reference = float(baseline.loc[REFERENCE_YEAR, "gdp_usd_bn"])
TARGET = {
    "capex_pct_gdp": 100.0 * capex_usd_bn / gdp_reference,
    "receipts_pct_gdp": float(window_share.mean()),
}
transport_record = ctx.read_json("thailand_transport")
assert abs(transport_record["target"]["capex_usd_bn"] - capex_usd_bn) < 1e-6, "the cost differs from the one used by the transport notebook"

print(f"Target economy: {ctx.target_name}; proposal status: {ctx.proposal.get('status', 'not stated')}")
print(f"Primary scenario {scenario['name']}: THB {scenario['capex_thb_bn']:g} billion at {fx_reference:.3f} THB per US$ ({REFERENCE_YEAR}) = US$ {capex_usd_bn:.3f} billion")
print(f"Cost as a share of GDP in {REFERENCE_YEAR}: {TARGET['capex_pct_gdp']:.3f} percent; receipts as a share of GDP, mean of {WINDOW_YEARS[0]} to {WINDOW_YEARS[-1]}: {TARGET['receipts_pct_gdp']:.3f} percent")
print("\nBaseline years used for the scaling")
show(baseline.loc[list(BASELINE_YEARS)], 3)

## Importance route

The predictive distribution of the importance route is built from the leave-one-economy-out prediction errors of the transport rule that the previous notebook validated. Each draw is the transported effect minus one such error, with the economies resampled, so the distribution describes the effect that a new study of an economy outside the sample would observe, including the systematic bias and the dispersion of the rule across economies. It does not use the uncertainty of the transported mean. That uncertainty, the sampling distribution of the transported effect from the cluster bootstrap over economies, is reported as a sensitivity row of the route tables; it describes the mean effect and not the effect of one new opening.

In [ ]:
transport_draws = ctx.read_table("thailand_transport_draws")
null_draws = ctx.read_table("thailand_null_draws")
PREDICTIVE_METHOD = str(transport_record["predictive"]["method"])
OT_PP = transport_draws["predictive_pp"].to_numpy(dtype=float)
OT_REL = transport_draws["predictive_rel_pct"].to_numpy(dtype=float)
OT_SAMPLING_PP = transport_draws["transport_pp"].to_numpy(dtype=float)
OT_SAMPLING_REL = transport_draws["transport_rel_pct"].to_numpy(dtype=float)
print(
    f"Predictive draws of the importance route: {len(transport_draws)}, from the leave-one-economy-out errors of {PREDICTIVE_METHOD} "
    f"({transport_record['predictive']['n_episodes']} episodes in {transport_record['predictive']['n_economies']} economies); null draws: {len(null_draws)}"
)
for label, pp, rel in (("Predictive draws", OT_PP, OT_REL), ("Sampling distribution of the transported effect", OT_SAMPLING_PP, OT_SAMPLING_REL)):
    print(f"{label}")
    print(f"  effect in percentage points of GDP: median {np.median(pp):.4f}, 5th to 95th percentile {np.quantile(pp, 0.05):.4f} to {np.quantile(pp, 0.95):.4f}")
    print(f"  effect in percent of the counterfactual receipts: median {np.median(rel):.2f}, 5th to 95th percentile {np.quantile(rel, 0.05):.2f} to {np.quantile(rel, 0.95):.2f}")

## Ambient routes

The ambient routes use the primary episodes that have a value for every covariate, and the placebo standard deviation of each effect as its standard error. Imputed covariate values are taken from the imputed feature table, and a covariate that the imputation dropped is taken from the observed feature table. A1 is the Bayesian normal hierarchical model, whose posterior predictive distribution is the pooled effect of a new opening; its mean effect has the default normal prior of the module, centred at zero and wide on the scale of the data, and the prior in use is printed with the fit. A2 is the scaling law through the origin in the cost as a share of GDP. A3 is the mixed-effects meta-regression on the cost share and the receipts share, with a ridge penalty on the standardised slopes. The two regression routes give Student predictive distributions whose degrees of freedom are printed with each fit; when they are below one the interval of the route is unavailable, a message says so and the route is left out of the draws. The covariates of the target are matched to those of the episodes by name.

In [ ]:
features_observed = ctx.read_table("case_features").set_index("case_id")
features_imputed = ctx.read_table("case_features_imputed").set_index("case_id")
frame = effects.copy()
for name in A3_FEATURES:
    filled = features_imputed if name in features_imputed.columns else features_observed
    frame[f"{name}_observed"] = frame["case_id"].map(features_observed[name])
    frame[name] = frame["case_id"].map(filled[name])
    print(f"Covariate {name}: taken from the {'imputed' if name in features_imputed.columns else 'observed'} feature table")
NEEDED = ["att_pp", "att_rel_pct", "placebo_sd_pp", "placebo_sd_rel_pct", *A3_FEATURES]
flagged = frame[frame["sample_primary"].astype(bool)]
AMB = flagged[np.isfinite(flagged[NEEDED].to_numpy(dtype=float)).all(axis=1)].reset_index(drop=True)
assert len(AMB) >= 6, f"the ambient routes need at least six primary episodes with complete covariates; found {len(AMB)}"
assert (AMB["placebo_sd_pp"] > 0).all() and (AMB["placebo_sd_rel_pct"] > 0).all(), "standard errors must be positive"
GROUPS = AMB["economy_group"].to_numpy()
n_imputed = {name: int(AMB[f"{name}_observed"].isna().sum()) for name in A3_FEATURES}
print(f"Primary episodes flagged: {len(flagged)}; with complete values for the ambient routes: {len(AMB)} in {len(set(GROUPS))} economies; values imputed: {n_imputed}")
print(f"Cost share of GDP in the sample: from {AMB['capex_pct_gdp'].min():.3f} to {AMB['capex_pct_gdp'].max():.3f} percent; target {TARGET['capex_pct_gdp']:.3f} percent, {TARGET['capex_pct_gdp'] / AMB['capex_pct_gdp'].max():.2f} times the largest case")
print(f"Receipts share of GDP in the sample: from {AMB['receipts_pct_gdp'].min():.3f} to {AMB['receipts_pct_gdp'].max():.3f} percent; target {TARGET['receipts_pct_gdp']:.3f} percent")

### Selection of the ambient estimator

The selection rule is fixed before the numbers below are computed. Each estimator is refitted without every episode of the held-out economy and predicts the held-out episodes, and the RMSE of these predictions is compared. A2 or A3 becomes the ambient estimator only if its RMSE is below the RMSE of A1 by the margin printed below; if both qualify, the one with the smaller RMSE is chosen; otherwise A1 is the ambient estimator. The comparison uses the effect in percentage points of GDP and the selected estimator is applied to both outcomes.

In [ ]:
ESTIMATORS = ("A1", "A2", "A3")
DESCRIPTION = {
    "A1": "hierarchical normal model, pooled posterior predictive",
    "A2": "scaling law in the cost share of GDP",
    "A3": "meta-regression on the cost share and the receipts share",
}
print(f"Rule: A2 or A3 replaces A1 only if its RMSE is at most {SELECTION_RATIO:g} times the RMSE of A1 (at least {100 * (1 - SELECTION_RATIO):.0f} percent lower)")
held_out = {name: np.full(len(AMB), np.nan) for name in ESTIMATORS}
for economy in sorted(set(GROUPS)):
    test_mask = GROUPS == economy
    assert int((~test_mask).sum()) >= 3, "too few episodes outside the held-out economy"
    for name in ESTIMATORS:
        held_out[name][test_mask] = predict_held_out(name, AMB[~test_mask], AMB[test_mask], "att_pp", "placebo_sd_pp")
    if ctx.guard:
        ctx.guard()
observed = AMB["att_pp"].to_numpy(dtype=float)
loeo = pd.DataFrame(
    {
        "estimator": list(ESTIMATORS),
        "description": [DESCRIPTION[name] for name in ESTIMATORS],
        "rmse": [float(np.sqrt(np.mean((observed - held_out[name]) ** 2))) for name in ESTIMATORS],
        "mae": [float(np.mean(np.abs(observed - held_out[name]))) for name in ESTIMATORS],
        "mean_error": [float(np.mean(observed - held_out[name])) for name in ESTIMATORS],
    }
).set_index("estimator", drop=False)
loeo["ratio_to_A1"] = loeo["rmse"] / loeo.loc["A1", "rmse"]
loeo["qualifies"] = (loeo.index != "A1") & (loeo["ratio_to_A1"] <= SELECTION_RATIO)
qualifying = loeo[loeo["qualifies"]]
SELECTED = "A1" if qualifying.empty else str(qualifying["rmse"].idxmin())
loeo["selected"] = loeo.index == SELECTED
loeo["n_episodes"] = len(AMB)
loeo["n_economies"] = len(set(GROUPS))
show(loeo.drop(columns=["estimator"]), 4)
print(f"\nAmbient estimator selected by the rule: {SELECTED} ({DESCRIPTION[SELECTED]})")
print(
    f"For reference, leave-one-economy-out RMSE of the transport predictors, with the importance weights relearned in every fold "
    f"({n_validated} validated episodes; {len(AMB)} episodes in the ambient validation):"
)
print(loco_summary["rmse"].round(4).to_string())
saved_loeo = ctx.save_table(loeo.reset_index(drop=True), "ambient_loeo")

route_record["ambient_selection"] = {
    "rule": "A2 or A3 replaces A1 only if its leave-one-economy-out RMSE is at least 5 percent below that of A1; the smaller RMSE wins if both qualify",
    "selection_ratio": SELECTION_RATIO,
    "outcome": "att_pp",
    "rmse": {name: float(loeo.loc[name, "rmse"]) for name in ESTIMATORS},
    "ratio_to_A1": {name: float(loeo.loc[name, "ratio_to_A1"]) for name in ESTIMATORS},
    "selected": SELECTED,
    "n_episodes": int(len(AMB)),
    "n_economies": int(len(set(GROUPS))),
}
saved_route = ctx.save_json(route_record, "route_decision")

### Predictions of the ambient estimators for the target

Every estimator is fitted to the whole primary sample for both outcomes. The covariates of the target are the cost share and the receipts share printed above.

In [ ]:
AMBIENT: dict[str, dict[str, dict[str, object]]] = {}
for i, name in enumerate(ESTIMATORS):
    AMBIENT[name] = {
        "pp": fit_ambient(name, AMB, "att_pp", "placebo_sd_pp", TARGET, SEED_META + 10 * i, label=f"{name}, effect in percentage points of GDP"),
        "rel": fit_ambient(name, AMB, "att_rel_pct", "placebo_sd_rel_pct", TARGET, SEED_META + 10 * i + 1, label=f"{name}, effect in percent of the counterfactual receipts"),
    }
    if ctx.guard:
        ctx.guard()
AVAILABLE = [name for name in ESTIMATORS if all(AMBIENT[name][outcome]["draws"] is not None for outcome in ("pp", "rel"))]
print(f"Estimators with a predictive interval: {AVAILABLE}; left out of the draws: {[name for name in ESTIMATORS if name not in AVAILABLE]}")
assert SELECTED in AVAILABLE, f"the selected ambient estimator {SELECTED} has no predictive interval"
prediction_rows = []
for name in AVAILABLE:
    for outcome, label in (("pp", "att_pp"), ("rel", "att_rel_pct")):
        result = AMBIENT[name][outcome]
        q = np.quantile(result["draws"], PROBS)
        prediction_rows.append({"estimator": name, "outcome": label, "mean": result["mean"], "q5": q[0], "q25": q[1], "q50": q[2], "q75": q[3], "q95": q[4]})
prediction_table = pd.DataFrame(prediction_rows)
show(prediction_table, 4)

a1_fit = AMBIENT["A1"]["pp"]["fit"]
a1_fit_rel = AMBIENT["A1"]["rel"]["fit"]
print("\nA1 on the effect in percentage points of GDP (posterior summary)")
show(a1_fit.summary, 4)
print(
    f"Prior of the mean effect: normal with mean {a1_fit.mu_prior[0]:g} and standard deviation {a1_fit.mu_prior[1]:.4f} (percentage points of GDP) and "
    f"{a1_fit_rel.mu_prior[1]:.4f} (percent of the counterfactual); prior of the between-episode standard deviation: half-normal with scale "
    f"{a1_fit.tau_prior_scale:.4f} and {a1_fit_rel.tau_prior_scale:.4f}"
)
print(f"Posterior probability that the predictive effect is positive: {a1_fit.prob_pred_positive:.3f}")
a2_fit = AMBIENT["A2"]["pp"]["fit"]
print(f"\nA2 on the effect in percentage points of GDP: slope {a2_fit['beta']:.4f} percentage points per percentage point of GDP of cost (standard error {a2_fit['se_beta']:.4f}); between-episode sd {np.sqrt(a2_fit['tau2']):.4f}; degrees of freedom {a2_fit['df']}")
a3_fit = AMBIENT["A3"]["pp"]["fit"]
print("\nA3 on the effect in percentage points of GDP: coefficients on standardised features")
show(a3_fit["coef"], 4)
print(f"Between-episode sd {np.sqrt(a3_fit['tau2']):.4f}; degrees of freedom {a3_fit['df']}")

## Scaling to US dollars and baht

The absolute scaling multiplies the effect in percentage points of GDP by the GDP of the baseline year. The relative scaling multiplies the effect in percent of the counterfactual receipts by the receipts of the baseline year. Both give the annual change in receipts in US$ billion, and the exchange rate of the baseline year converts it to THB billion. The route table has one block for the importance route, one for its sampling-only sensitivity, one for the selected ambient estimator under the name ambient, and one for each ambient estimator under its own name.

In [ ]:
ROUTE_DRAWS: dict[str, dict[str, np.ndarray]] = {
    "ot_importance": {"pp": OT_PP, "rel": OT_REL},
    "ot_importance_sampling": {"pp": OT_SAMPLING_PP, "rel": OT_SAMPLING_REL},
    "ambient": {"pp": AMBIENT[SELECTED]["pp"]["draws"], "rel": AMBIENT[SELECTED]["rel"]["draws"]},
}
for name in AVAILABLE:
    ROUTE_DRAWS[f"ambient_{name}"] = {"pp": AMBIENT[name]["pp"]["draws"], "rel": AMBIENT[name]["rel"]["draws"]}
ESTIMATOR_OF = {"ot_importance": "transport", "ot_importance_sampling": "transport_sampling", "ambient": SELECTED, **{f"ambient_{name}": name for name in AVAILABLE}}

tables = {
    route: impact.impact_table(draws["pp"], draws["rel"], baseline, years=BASELINE_YEARS, route=route, probs=PROBS)
    for route, draws in ROUTE_DRAWS.items()
}
comparison = impact.route_comparison(tables, decision)
comparison["estimator"] = comparison["route"].map(ESTIMATOR_OF)
fx_by_year = baseline["fx_thb_per_usd"]
for level in ("q5", "q50", "q95"):
    comparison[f"{level}_thb_bn"] = comparison[level] * comparison["baseline_year"].map(fx_by_year)
print(f"Annual change in international tourism receipts, US$ billion (primary route: {decision.primary})")
show(comparison[["route", "estimator", "baseline_year", "scaling", "q5", "q25", "q50", "q75", "q95", "median_share_receipts_pct", "median_share_gdp_pct", "primary"]], 3)
saved_impact = ctx.save_table(comparison, "thailand_impact")

## Detection

The transported placebo draws of the previous notebook describe what the method reports where nothing happened. The detection threshold is the quantile of their absolute values at one minus the size of the test. The minimum detectable effect is the threshold plus the larger of the placebo quantile at the power and the negative of the placebo quantile at one minus the power, so that an effect of that size and of either sign exceeds the threshold in at least the stated share of the placebo draws; the empirical placebo distribution is used as it is, and the normal approximation, the threshold plus the standard normal quantile of the power times the standard deviation of the placebo draws, is reported next to it. The predictive draws of the importance route are observed effects, because they carry the prediction errors of earlier openings, so the detection probability of that route is the share of its draws beyond the threshold. The draws of the ambient routes describe the true effect of a new opening, so each draw is observed with an independent draw from the placebo distribution before it is compared with the threshold. The size and the power are printed below.

In [ ]:
NULLS = {"pp": null_draws["null_pp"].to_numpy(dtype=float), "rel": null_draws["null_rel_pct"].to_numpy(dtype=float)}
UNITS = {"pp": "percentage points of GDP", "rel": "percent of the counterfactual receipts"}
SCALING_OF = {"pp": "absolute", "rel": "relative"}
TRUE_EFFECT_ROUTES = ("ambient", *(f"ambient_{name}" for name in AVAILABLE))
DETECTION_ROUTES = [route for route in ROUTE_DRAWS if route != "ot_importance_sampling"]
detection: dict[str, dict[str, float]] = {}
print(f"Size of the test {ALPHA}; power {POWER}; standard normal quantile of the power {stats.norm.ppf(POWER):.4f}")
for outcome in ("pp", "rel"):
    limits = impact.minimum_detectable_effect(NULLS[outcome], alpha=ALPHA, power=POWER)
    detection[outcome] = {
        "threshold": limits["threshold"],
        "mde": limits["mde"],
        "mde_normal": limits["mde_normal"],
        "null_sd": float(NULLS[outcome].std(ddof=1)),
        "n_null_draws": int(len(NULLS[outcome])),
    }
    print(
        f"{SCALING_OF[outcome].capitalize()} scaling ({UNITS[outcome]}): placebo sd {detection[outcome]['null_sd']:.4f}; "
        f"detection threshold {limits['threshold']:.4f}; minimum detectable effect {limits['mde']:.4f} from the empirical placebo distribution "
        f"and {limits['mde_normal']:.4f} from the normal approximation"
    )


def route_detection_probability(route: str, outcome: str) -> float:
    """Probability that the observed effect of a route exceeds the detection threshold.

    Parameters
    ----------
    route : str
        Key of ``ROUTE_DRAWS``.
    outcome : str
        ``pp`` or ``rel``.

    Returns
    -------
    float
        The share of the draws of the route beyond the threshold; for a route in ``TRUE_EFFECT_ROUTES`` each draw is
        first observed with an independent placebo draw.
    """
    noise = NULLS[outcome] if route in TRUE_EFFECT_ROUTES else None
    return impact.detection_probability(ROUTE_DRAWS[route][outcome], detection[outcome]["threshold"], null_draws=noise)


detection_rows = []
for route in DETECTION_ROUTES:
    for outcome in ("pp", "rel"):
        median_effect = float(np.median(ROUTE_DRAWS[route][outcome]))
        detection_rows.append(
            {
                "route": route,
                "scaling": SCALING_OF[outcome],
                "median_effect": median_effect,
                "threshold": detection[outcome]["threshold"],
                "mde": detection[outcome]["mde"],
                "mde_normal": detection[outcome]["mde_normal"],
                "median_to_mde": median_effect / detection[outcome]["mde"],
                "placebo_noise_added": route in TRUE_EFFECT_ROUTES,
                "detection_probability": route_detection_probability(route, outcome),
            }
        )
detection_table = pd.DataFrame(detection_rows)
print("\nDetection by route (effects in percentage points of GDP for the absolute and percent of the counterfactual for the relative scaling)")
show(detection_table, 4)
print("\nThreshold and minimum detectable effect in US$ billion per year")
usd_rows = []
for year in BASELINE_YEARS:
    row = baseline.loc[year]
    for outcome, base in (("pp", row["gdp_usd_bn"]), ("rel", row["receipts_usd_bn"])):
        usd_rows.append(
            {
                "baseline_year": year,
                "scaling": SCALING_OF[outcome],
                "threshold_usd_bn": detection[outcome]["threshold"] / 100.0 * base,
                "mde_usd_bn": detection[outcome]["mde"] / 100.0 * base,
                "mde_normal_usd_bn": detection[outcome]["mde_normal"] / 100.0 * base,
                "mde_share_of_receipts_pct": 100.0 * detection[outcome]["mde"] / 100.0 * base / row["receipts_usd_bn"],
                "mde_share_of_gdp_pct": 100.0 * detection[outcome]["mde"] / 100.0 * base / row["gdp_usd_bn"],
            }
        )
detection_usd = pd.DataFrame(usd_rows)
show(detection_usd, 3)

## Reference comparison with the claims of officials

The proposal file records claims that officials have made about the annual benefit of the park. They are shown only as a reference scale. The file states that no study is cited, so the claims are not estimates from a study and the analysis does not test them.

In [ ]:
claims = ctx.proposal.get("ministerial_claims") or {}
CLAIM_SHARE = claims.get("share_of_gdp_per_year_percent")
if claims:
    print("Claims recorded in the proposal file:", {k: v for k, v in claims.items() if k != "note"})
    print("Note in the file:", claims.get("note", "none"))
else:
    print("The proposal file records no claims.")
claim_rows = []
if CLAIM_SHARE is not None:
    for year in BASELINE_YEARS:
        row = baseline.loc[year]
        claim_usd = float(CLAIM_SHARE) / 100.0 * row["gdp_usd_bn"]
        claim_rows.append({"baseline_year": year, "claim_usd_bn": claim_usd, "claim_thb_bn": claim_usd * row["fx_thb_per_usd"], "claim_share_of_receipts_pct": 100.0 * claim_usd / row["receipts_usd_bn"]})
    claim_table = pd.DataFrame(claim_rows).set_index("baseline_year")
    print(f"\nA claim of {CLAIM_SHARE:g} percent of GDP a year in US$ billion and THB billion, and as a share of the receipts of the baseline year")
    show(claim_table, 3)
    primary_draws = ROUTE_DRAWS[decision.primary]
    for year in BASELINE_YEARS:
        row = baseline.loc[year]
        absolute_usd = primary_draws["pp"] / 100.0 * row["gdp_usd_bn"]
        relative_usd = primary_draws["rel"] / 100.0 * row["receipts_usd_bn"]
        claim_usd = float(CLAIM_SHARE) / 100.0 * row["gdp_usd_bn"]
        print(
            f"Baseline {year}, primary route {decision.primary}: share of predictive draws at or above the claim {100 * np.mean(absolute_usd >= claim_usd):.1f} percent "
            f"(absolute scaling) and {100 * np.mean(relative_usd >= claim_usd):.1f} percent (relative scaling)"
        )
    print("The claim is a reference scale only and has no study behind it.")

## Summary of the primary route

The summary file holds, for the primary route and each scaling and baseline year, the median and the 5th and 95th percentiles of the annual change in receipts in US$ billion and THB billion, as a share of the baseline receipts and of the baseline GDP, together with the detection threshold, the minimum detectable effect from the empirical placebo distribution and from the normal approximation, and the detection probability. It also holds the percentiles of the sampling-only sensitivity of the importance route.

In [ ]:
primary_draws = ROUTE_DRAWS[decision.primary]
summary: dict[str, object] = {
    "primary_route": decision.primary,
    "primary_estimator": ESTIMATOR_OF[decision.primary],
    "predictive_method": PREDICTIVE_METHOD,
    "route_passed": {name: bool(flag) for name, flag in decision.passed.items()},
    "target": {
        "name": ctx.target_name,
        "scenario": scenario["name"],
        "capex_thb_bn": float(scenario["capex_thb_bn"]),
        "reference_year": REFERENCE_YEAR,
        "fx_thb_per_usd": fx_reference,
        "capex_usd_bn": capex_usd_bn,
        "capex_pct_gdp": TARGET["capex_pct_gdp"],
        "receipts_pct_gdp": TARGET["receipts_pct_gdp"],
    },
    "effect": {
        "att_pp": quantile_dict(primary_draws["pp"]),
        "att_rel_pct": quantile_dict(primary_draws["rel"]),
    },
    "baseline_years": [int(y) for y in BASELINE_YEARS],
    "by_baseline_year": {},
    "detection": {},
    "ambient_selected": SELECTED,
    "sensitivity": {
        "ot_importance_sampling": {
            "att_pp": quantile_dict(ROUTE_DRAWS["ot_importance_sampling"]["pp"]),
            "att_rel_pct": quantile_dict(ROUTE_DRAWS["ot_importance_sampling"]["rel"]),
        }
    },
}
for year in BASELINE_YEARS:
    row = baseline.loc[year]
    block = scaled_summary(primary_draws["pp"], primary_draws["rel"], row)
    block["baseline"] = {"gdp_usd_bn": float(row["gdp_usd_bn"]), "receipts_usd_bn": float(row["receipts_usd_bn"]), "fx_thb_per_usd": float(row["fx_thb_per_usd"])}
    summary["by_baseline_year"][str(year)] = block
for outcome in ("pp", "rel"):
    scaling = SCALING_OF[outcome]
    summary["detection"][scaling] = {
        "unit": UNITS[outcome],
        "threshold": detection[outcome]["threshold"],
        "minimum_detectable_effect": detection[outcome]["mde"],
        "mde_normal": detection[outcome]["mde_normal"],
        "detection_probability": route_detection_probability(decision.primary, outcome),
        "placebo_noise_added": decision.primary in TRUE_EFFECT_ROUTES,
        "null_sd": detection[outcome]["null_sd"],
        "alpha": ALPHA,
        "power": POWER,
    }
if CLAIM_SHARE is not None:
    summary["claim_reference"] = {"share_of_gdp_per_year_percent": float(CLAIM_SHARE), "note": claims.get("note"), "status": "reference only"}
saved_summary = ctx.save_json(summary, "thailand_summary")

## Figures

The first figure shows the annual change in receipts in US$ billion for every route and both scalings, for each baseline year. The thick bar is the 50 percent interval, the thin bar the 90 percent interval and the marker the median; the primary route has filled markers and the claim, when the proposal file states one, is the dashed vertical line. The importance route is shown with its predictive distribution and, as a sensitivity, with the sampling distribution of the transported effect alone. The second figure shows the case effects against the cost share of GDP with the fitted scaling law and its prediction for the target economy.

In [ ]:
ROUTE_LABEL = {
    "ot_importance": "Importance-weighted transport, predictive",
    "ot_importance_sampling": "Importance-weighted transport, sampling only",
    "ambient_A1": "Ambient A1, pooled hierarchical",
    "ambient_A2": "Ambient A2, scaling law in cost",
    "ambient_A3": "Ambient A3, meta-regression",
}
ROUTE_ORDER = [route for route in ROUTE_LABEL if route in tables]
PRIMARY_KEY = "ot_importance" if decision.primary == "ot_importance" else f"ambient_{SELECTED}"
fig, axes = plt.subplots(1, len(BASELINE_YEARS), figsize=(6.2 * len(BASELINE_YEARS), 0.5 * 2 * len(ROUTE_ORDER) + 1.2), sharey=True)
axes = np.atleast_1d(axes)
labels = []
for route in ROUTE_ORDER:
    for scaling in ("absolute", "relative"):
        labels.append(f"{ROUTE_LABEL[route]}, {scaling}" + (" (primary)" if route == PRIMARY_KEY else ""))
ypos = np.arange(len(labels))[::-1]
for ax, year in zip(axes, BASELINE_YEARS):
    k = 0
    for route in ROUTE_ORDER:
        table = tables[route]
        for scaling, marker in (("absolute", "o"), ("relative", "s")):
            r = table[(table["baseline_year"] == year) & (table["scaling"] == scaling)].iloc[0]
            y = ypos[k]
            ax.hlines(y, r["q25"], r["q75"], color="0.62", linewidth=7.0)
            ax.hlines(y, r["q5"], r["q95"], color="0.0", linewidth=1.3)
            ax.plot(
                r["q50"], y, marker=marker, markersize=8, linestyle="none", markeredgecolor="0.0",
                markerfacecolor="0.0" if route == PRIMARY_KEY else "white", zorder=4,
            )
            k += 1
    ax.axvline(0.0, color="0.0", linewidth=0.8)
    if CLAIM_SHARE is not None:
        claim_usd = float(CLAIM_SHARE) / 100.0 * baseline.loc[year, "gdp_usd_bn"]
        ax.axvline(claim_usd, color="0.2", linestyle="--", linewidth=1.3)
    ax.set_title(f"Baseline {year}")
    ax.set_xlabel("Annual change in receipts (US$ billion per year)")
    ax.set_yticks(ypos)
    ax.set_yticklabels(labels, fontsize=8)
    ax.grid(axis="y", visible=False)
axes[0].set_ylabel("Route and scaling")
handles = [
    Line2D([], [], color="0.62", linewidth=7.0, label="50 percent interval"),
    Line2D([], [], color="0.0", linewidth=1.3, label="90 percent interval"),
    Line2D([], [], color="0.0", marker="o", markerfacecolor="white", linestyle="none", label="Median, absolute scaling"),
    Line2D([], [], color="0.0", marker="s", markerfacecolor="white", linestyle="none", label="Median, relative scaling"),
    Line2D([], [], color="0.0", marker="o", markerfacecolor="0.0", linestyle="none", label="Primary route (filled)"),
]
if CLAIM_SHARE is not None:
    handles.append(Line2D([], [], color="0.2", linestyle="--", linewidth=1.3, label=f"Claim of {CLAIM_SHARE:g} percent of GDP a year (reference only)"))
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=3, fontsize=8)
fig.suptitle("Annual change in international tourism receipts by route", y=1.0)
fig.tight_layout()
figure_path = ctx.save_figure(fig, "impact_routes.png")

In [ ]:
assert "A2" in AVAILABLE, "the scaling-law figure needs the prediction interval of A2"
x_case = AMB["capex_pct_gdp"].to_numpy(dtype=float)
y_case = AMB["att_pp"].to_numpy(dtype=float)
se_case = AMB["placebo_sd_pp"].to_numpy(dtype=float)
a2 = AMBIENT["A2"]["pp"]
t_level = float(stats.t.ppf(0.5 + LEVEL / 2.0, a2_fit["df"]))


def band(x: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Fitted scaling law and the edges of its prediction interval at the sizes ``x``."""
    centre = a2_fit["beta"] * x
    half = t_level * np.sqrt(a2_fit["tau2"] + x**2 * a2_fit["se_beta"] ** 2)
    return centre, centre - half, centre + half


fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.8))
x_ranges = (
    ("Range of the case costs", np.linspace(0.0, 1.08 * x_case.max(), 200)),
    ("Range extended to the target cost", np.linspace(0.0, 1.08 * max(x_case.max(), TARGET["capex_pct_gdp"]), 200)),
)
for ax, (title, grid) in zip(axes, x_ranges):
    centre, lower, upper = band(grid)
    ax.fill_between(grid, lower, upper, color="0.88", label=f"{100 * LEVEL:.0f} percent prediction interval of the scaling law")
    ax.plot(grid, centre, color="0.0", linestyle="-", linewidth=1.5, label="Fitted scaling law through the origin")
    ax.errorbar(x_case, y_case, yerr=se_case, fmt="o", color="0.0", markerfacecolor="white", markersize=6, capsize=2, linewidth=0.9, label="Case effect (bars: one placebo standard deviation)", zorder=3)
    ax.errorbar(
        [TARGET["capex_pct_gdp"]], [a2["mean"]], yerr=[[a2["mean"] - a2["lo"]], [a2["hi"] - a2["mean"]]],
        fmt="D", color="0.0", markersize=8, capsize=4, linewidth=1.6, label=f"Target: prediction and {100 * LEVEL:.0f} percent interval", zorder=4,
    )
    ax.axhline(0.0, color="0.0", linewidth=0.8)
    ax.set_xlim(left=0.0)
    ax.set_xlabel("Cost of the park as a share of GDP (percent of GDP)")
    ax.set_ylabel("Effect on receipts (percentage points of GDP)")
    ax.set_title(title)
handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=2, fontsize=8)
fig.suptitle("Case effects and the scaling law in the cost share of GDP", y=1.0)
fig.tight_layout()
figure_path = ctx.save_figure(fig, "impact_scaling_law.png")

## Conclusions

The sentences below are produced from the results above.

In [ ]:
CRITERION_NAME = {"importance_usable": "importance model usable", "loco_rmse": "validation RMSE criterion", "enough_cases": "enough validated episodes"}
primary_label = decision.primary if decision.primary == "ot_importance" else f"ambient ({SELECTED}, {DESCRIPTION[SELECTED]})"
other_key = "ambient" if decision.primary == "ot_importance" else "ot_importance"
print(
    f"The rule that was fixed in advance selects the {primary_label} route as primary. Criteria met: "
    + ", ".join(f"{CRITERION_NAME[name]} {'yes' if flag else 'no'}" for name, flag in decision.passed.items())
    + f". The validation relearned the importance weights in {n_folds - n_uniform_folds} of {n_folds} folds."
)
for year in BASELINE_YEARS:
    block = summary["by_baseline_year"][str(year)]
    for scaling in ("absolute", "relative"):
        b = block[scaling]
        print(
            f"With the baseline of {year} and the {scaling} scaling, the primary route puts the annual change in international tourism receipts at a median of "
            f"US$ {b['usd_bn']['q50']:.2f} billion (THB {b['thb_bn']['q50']:.0f} billion), with a 90 percent interval from US$ {b['usd_bn']['q5']:.2f} to {b['usd_bn']['q95']:.2f} billion. "
            f"The median is {b['share_of_receipts_pct']['q50']:.1f} percent of the receipts and {b['share_of_gdp_pct']['q50']:.2f} percent of the GDP of {year}."
        )
for outcome in ("pp", "rel"):
    scaling = SCALING_OF[outcome]
    d = summary["detection"][scaling]
    print(
        f"On the {scaling} scaling the transported placebo draws give a detection threshold of {d['threshold']:.3f} and a minimum detectable effect of {d['minimum_detectable_effect']:.3f} {UNITS[outcome]} "
        f"({d['mde_normal']:.3f} with the normal approximation); the probability that the observed effect of the primary route exceeds the threshold is {100 * d['detection_probability']:.1f} percent."
    )
other = comparison[(comparison["route"] == other_key) & (comparison["baseline_year"] == BASELINE_YEARS[0]) & (comparison["scaling"] == "absolute")].iloc[0]
primary_row = comparison[(comparison["primary"]) & (comparison["baseline_year"] == BASELINE_YEARS[0]) & (comparison["scaling"] == "absolute")].iloc[0]
sampling = comparison[(comparison["route"] == "ot_importance_sampling") & (comparison["baseline_year"] == BASELINE_YEARS[0]) & (comparison["scaling"] == "absolute")].iloc[0]
print(
    f"For comparison, the {other_key} route gives a median of US$ {other['q50']:.2f} billion (90 percent interval {other['q5']:.2f} to {other['q95']:.2f}) on the absolute scaling "
    f"with the baseline of {BASELINE_YEARS[0]}, against US$ {primary_row['q50']:.2f} billion for the primary route."
)
print(
    f"The importance route takes its predictive distribution from the leave-one-economy-out errors of {PREDICTIVE_METHOD}. The sampling distribution of the transported effect alone gives a 90 percent interval "
    f"from US$ {sampling['q5']:.2f} to {sampling['q95']:.2f} billion on the absolute scaling with the baseline of {BASELINE_YEARS[0]}."
)
if CLAIM_SHARE is not None:
    claim_first = claim_table.loc[BASELINE_YEARS[0]]
    print(
        f"The claim of {CLAIM_SHARE:g} percent of GDP a year corresponds to US$ {claim_first['claim_usd_bn']:.2f} billion with the baseline of {BASELINE_YEARS[0]}. "
        f"It is shown as a reference only; no study stands behind it."
    )
print(f"The cost of the proposal is {TARGET['capex_pct_gdp']:.2f} percent of GDP, {TARGET['capex_pct_gdp'] / AMB['capex_pct_gdp'].max():.2f} times the largest cost share among the {len(AMB)} primary episodes.")

## Checks

In [ ]:
assert decision.primary in ("ot_importance", "ambient")
EXPECTED_ROUTES = {"ot_importance", "ot_importance_sampling", "ambient", *(f"ambient_{name}" for name in AVAILABLE)}
assert set(comparison["route"]) == EXPECTED_ROUTES
assert comparison.groupby("route")["primary"].all().loc[decision.primary] and not comparison.groupby("route")["primary"].any().drop(decision.primary).any()
assert len(comparison) == len(EXPECTED_ROUTES) * len(BASELINE_YEARS) * 2
assert (comparison["q5"] <= comparison["q50"]).all() and (comparison["q50"] <= comparison["q95"]).all()
assert np.isfinite(comparison[["q5", "q25", "q50", "q75", "q95", "median_share_receipts_pct", "median_share_gdp_pct"]].to_numpy()).all()
assert set(loeo.index) == set(ESTIMATORS) and np.isfinite(loeo["rmse"]).all() and loeo["selected"].sum() == 1 and SELECTED in AVAILABLE
assert all(np.isfinite(ROUTE_DRAWS[route][o]).all() and len(ROUTE_DRAWS[route][o]) > 100 for route in ROUTE_DRAWS for o in ("pp", "rel"))

# The route rule recomputed from the saved validation summary
rmse = loco_summary["rmse"]
rule_met = (
    IMPORTANCE_USABLE
    and rmse["ot_weighted"] <= rules["rmse_ratio_equal"] * rmse["equal"]
    and rmse["ot_weighted"] <= rules["rmse_ratio_uniform"] * rmse["ot_uniform"]
    and n_validated >= rules["min_cases"]
)
assert (decision.primary == "ot_importance") == bool(rule_met)
assert 0 <= n_uniform_folds <= n_folds == n_validated_economies
assert np.allclose(fold_weights.groupby("held_out_group")["weight"].sum(), 1.0)

# The importance route uses the predictive draws of the previous notebook
for outcome, draws in (("att_pp", OT_PP), ("att_rel_pct", OT_REL)):
    stored = transport_record[outcome]["predictive_percentiles"]
    assert np.allclose([stored["p5"], stored["p50"], stored["p95"]], np.quantile(draws, [0.05, 0.5, 0.95]))
assert transport_record["att_pp"]["predictive_method"] == PREDICTIVE_METHOD == transport_record["predictive"]["method"]

# Detection: threshold, minimum detectable effect and probability recomputed directly from the placebo draws
for outcome in ("pp", "rel"):
    null = NULLS[outcome]
    threshold = float(np.quantile(np.abs(null), 1.0 - ALPHA))
    assert np.isclose(detection[outcome]["threshold"], threshold)
    assert detection[outcome]["mde"] >= detection[outcome]["threshold"] > 0 and np.isfinite(detection[outcome]["mde_normal"]) and detection[outcome]["mde_normal"] > 0
    atom = float(np.unique(null, return_counts=True)[1].max()) / len(null)
    for sign in (1.0, -1.0):
        assert np.mean(np.abs(sign * detection[outcome]["mde"] + null) > threshold) >= POWER - atom
    sample = slice(0, 500)
    for route in DETECTION_ROUTES:
        draws = ROUTE_DRAWS[route][outcome][sample]
        shifted = draws[:, None] + null[None, sample] if route in TRUE_EFFECT_ROUTES else draws[:, None]
        assert np.isclose(
            impact.detection_probability(draws, threshold, null_draws=null[sample] if route in TRUE_EFFECT_ROUTES else None),
            np.mean(np.abs(shifted) > threshold),
        )
assert all(0.0 <= row["detection_probability"] <= 1.0 for row in detection_rows) and {row["route"] for row in detection_rows} == set(DETECTION_ROUTES)

for name in ("route_decision.json", "thailand_summary.json"):
    assert (ctx.results_dir / name).is_file(), name
for name in ("thailand_impact", "ambient_loeo"):
    assert ctx.table_path(name).is_file(), name
saved_decision = ctx.read_json("route_decision")
assert saved_decision["primary"] == decision.primary and set(saved_decision) >= {"primary", "passed", "details", "inputs", "ambient_selection"}
assert saved_decision["inputs"]["n_validated_episodes"] == n_validated and "leave-one-economy-out" in saved_decision["details"]
saved_summary_check = ctx.read_json("thailand_summary")
assert saved_summary_check["primary_route"] == decision.primary and set(saved_summary_check["by_baseline_year"]) == {str(y) for y in BASELINE_YEARS}
assert saved_summary_check["predictive_method"] == PREDICTIVE_METHOD and set(saved_summary_check["sensitivity"]) == {"ot_importance_sampling"}
assert all("mde_normal" in saved_summary_check["detection"][scaling] for scaling in ("absolute", "relative"))
saved_impact_check = pd.read_csv(ctx.table_path("thailand_impact"))
assert saved_impact_check["primary"].dtype == bool and set(saved_impact_check["route"]) == EXPECTED_ROUTES
print("Checks passed.")
print(f"Elapsed time of the notebook: {time.perf_counter() - NOTEBOOK_START:.0f} seconds")

## Scope and limits

The estimates are conditional on a park of the stated cost opening. They are mean annual effects on international tourism receipts over the first five years after the opening, in percentage points of GDP and in percent of the counterfactual receipts, scaled to US dollars and baht with the baseline of the year named in each table. The importance route moves the estimated effects of earlier openings to the feature cloud of Thailand; it does not estimate the effect of a Thai park from Thai data, and the estimates are not the total economic impact of a park, which would add domestic visitors, employment, tax revenue and displacement of other spending.

The route rule compares leave-one-economy-out errors in which the importance weights are relearned inside every fold from the training economies, so the validation is not optimistic about the learning of the weights, and a fold whose weights could not be fitted is counted and uses uniform weights. The comparison is a point rule on the RMSE; the bootstrap intervals of the RMSE ratios in the previous notebook describe the cases at hand and are not a test of the rule. The predictive distribution of the importance route carries the leave-one-economy-out errors of the transport rule, which are few and are resampled by economy, so its tails rest on the observed errors only. The sampling distribution of the transported effect, shown as a sensitivity, describes the mean effect and not the effect of one new opening.

The case effects come from country-level synthetic controls with few pre-opening years, and the pandemic years are excluded from every estimation window. The proposal is not a commitment: the sources reviewed record no signed agreement, approved budget, named operator or site allocation. The cost in US dollars is derived from the cost in baht with the exchange rate of one year, and the dollar costs of the earlier cases are derived in the same way, so every cost is an estimate.

Cases in one economy are not independent. The validation of the ambient estimators leaves out whole economies, and the printed number of economies, not of episodes, is the effective number of independent observations. When the cost of the proposal as a share of GDP lies beyond the cost shares of the cases, the scaling law and the meta-regression extrapolate, and the printed ratio to the largest case reports how far. The detection analysis takes the placebo distribution of the transport method as the noise of an observed effect; the draws of the ambient routes are observed with a draw from it, and the draws of the importance route already carry the prediction errors. The claims of officials are a reference scale only, because no study stands behind them.